In [19]:
# Imports
import os
import pandas as pd
import asyncio
from typing import Literal
from dotenv import load_dotenv, find_dotenv
from google import genai
from google.genai import types
from pydantic import BaseModel, Field
from dotenv import load_dotenv, find_dotenv
import plotly.graph_objects as go
import plotly.express as px
import kaleido
from datetime import datetime
import json

In [20]:
load_dotenv(find_dotenv()) # Find env file and load API Key
api_key = os.getenv("GOOGLE_GEMINI_KEY") # Retrieve API Key from environment variable


In [21]:
client = genai.Client(api_key=api_key)

In [22]:
# Define the desired JSON output structure for binary classification (Negative/Positive)
class DDIClassification(BaseModel):
    classification: Literal[
        "Negative",
        "Positive"
    ] = Field(
        description="Binary classification: Negative = adverse effects/harmful to patient; Positive = beneficial/increased efficacy/non-threatening"
    )
    reasoning: str = Field(
        description="A short, clear one-sentence clinical explanation for the classification."
    )

In [23]:
# Looping through the dataframe, extracting the interaction descriptions, 
# then appending the interaciton description to the list of dictionaries in the column
df = pd.read_csv("C:\\Users\\ashto\\ddi-prediction\\data\\sample\\drugbank_approved_small_2369_1129743_ddi_pairs.csv") # Load in the csv

### ASYNC CODE

In [25]:
# Create async wrapper for binary classification with retry logic and rate limit handling
async def classify_interaction_async(description: str, max_retries: int = 5) -> DDIClassification:
    system_prompt = """
    You are an expert clinical pharmacologist analyzing Drug-Drug Interactions (DDIs) from DrugBank.
    Your task is to classify each DDI into ONE of two categories based on clinical impact.

    CLASSIFICATION RULES:
    
    "Negative" - Any adverse effect or harmful outcome for the patient:
    - Increased bleeding or hemorrhage risk
    - Toxicity to organs (liver, kidney, cardiac, etc.)
    - CNS/neurological effects (sedation, seizures, dizziness, etc.)
    - Cardiovascular problems (hypertension, hypotension, arrhythmias, etc.)
    - Blood disorders (neutropenia, methemoglobinemia, etc.)
    - Electrolyte imbalances
    - Metabolic dysregulation (hyperglycemia, hypoglycemia, etc.)
    - Allergic/hypersensitivity reactions
    - Loss of efficacy or reduced drug effectiveness
    - GI effects (nausea, vomiting, diarrhea, etc.)
    - Absorption/metabolic interference that reduces drug effectiveness
    - ANY vague mention of "adverse effects" or clinical harm
    - Requires dosage adjustment due to safety concerns
    
    "Positive" - Beneficial effects or non-threatening interactions:
    - Enhanced therapeutic efficacy
    - Synergistic beneficial effects
    - Positive drug interactions that improve treatment outcomes
    - No significant interaction (neutral/safe)
    - Requires monitoring only without specific adverse outcome
    
    DECISION LOGIC:
    - If the description mentions ANY potential harm, toxicity, adverse effect, or reduction in safety → "Negative"
    - If the description only mentions beneficial effects or no significant interaction → "Positive"
    - When in doubt about severity, classify as "Negative" (safety-first approach)
    """
    
    user_prompt = f"Classify this DDI as either 'Negative' (harmful/adverse effects) or 'Positive' (beneficial/non-threatening):\n\n{description}"
    
    for attempt in range(max_retries):
        try:
            # Run the synchronous API call in a thread pool to avoid blocking
            loop = asyncio.get_event_loop()
            result = await loop.run_in_executor(None, lambda: client.models.generate_content(
                model='gemini-2.5-flash',
                contents=user_prompt,
                config=types.GenerateContentConfig(
                    system_instruction=system_prompt,
                    temperature=0.0,
                    response_mime_type="application/json",
                    response_schema=DDIClassification,
                ),
            ))
            return DDIClassification.model_validate_json(result.text)
        except Exception as e:
            error_msg = str(e)
            is_rate_limit = "429" in error_msg or "quota" in error_msg.lower() or "high demand" in error_msg.lower()
            
            if attempt < max_retries - 1:
                # Exponential backoff with longer waits for rate limits
                if is_rate_limit:
                    wait_time = (2 ** attempt) * 5  # 5s, 10s, 20s, 40s, 80s for rate limits
                else:
                    wait_time = (2 ** attempt)  # 1s, 2s, 4s, 8s, 16s for other errors
                
                print(f"Attempt {attempt + 1}/{max_retries}: Waiting {wait_time}s... ({error_msg[:80]})")
                await asyncio.sleep(wait_time)
            else:
                print(f"Failed after {max_retries} attempts: {error_msg[:150]}")
                raise

In [ ]:
async def process_samples_async(sample_size: int = 1500, batch_size: int = 50, delay_between_batches: float = 1.0):
    """
    Process interactions with concurrent batching using binary classification.
    
    Args:
        sample_size: Total interactions to sample
        batch_size: Number of concurrent requests per batch
        delay_between_batches: Seconds to wait between batches (helps with rate limits)
    """
    df_sample = df.sample(n=sample_size, random_state=42).reset_index(drop=True)
    descriptions = df_sample['description'].tolist()
    
    print(f"Processing {len(descriptions)} interactions ({batch_size} concurrent per batch)...")
    
    # Process in batches to manage rate limits
    all_results = []
    num_batches = (len(descriptions) + batch_size - 1) // batch_size
    
    for batch_num in range(num_batches):
        start_idx = batch_num * batch_size
        end_idx = min((batch_num + 1) * batch_size, len(descriptions))
        batch_descs = descriptions[start_idx:end_idx]
        
        print(f"  Batch {batch_num + 1}/{num_batches}: Processing {len(batch_descs)} items...")
        
        # Create concurrent tasks for this batch
        tasks = [classify_interaction_async(desc) for desc in batch_descs]
        batch_results = await asyncio.gather(*tasks, return_exceptions=True)
        all_results.extend(batch_results)
        
        # Add delay between batches (except after last batch)
        if batch_num < num_batches - 1:
            await asyncio.sleep(delay_between_batches)
    
    # Assign results to dataframe
    for idx, result in enumerate(all_results):
        if isinstance(result, Exception):
            print(f"Error at index {idx}: {str(result)[:100]}")
            df_sample.loc[idx, 'classification'] = 'ERROR'
            df_sample.loc[idx, 'reasoning'] = str(result)[:200]
        else:
            df_sample.loc[idx, 'classification'] = result.classification
            df_sample.loc[idx, 'reasoning'] = result.reasoning
    
    # Save to CSV
    output_path = f"C:\\Users\\ashto\\ddi-prediction\\data\\sample\\drug_class_binary_1500.csv"
    df_sample.to_csv(output_path, index=False)
    print(f"\n✓ Binary classification saved to {output_path}")
    
    return df_sample

# Run with optimized settings for paid tier
try:
    loop = asyncio.get_running_loop()
except RuntimeError:
    df_sample_paid = asyncio.run(process_samples_async(sample_size=1500, batch_size=50, delay_between_batches=0.5))
else:
    df_sample_paid = await process_samples_async(sample_size=1500, batch_size=50, delay_between_batches=0.5)

Processing 1500 interactions (50 concurrent per batch)...
  Batch 1/30: Processing 50 items...
  Batch 2/30: Processing 50 items...
Attempt 1/5: Waiting 5s... (503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently exp)
Attempt 1/5: Waiting 5s... (503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently exp)
Attempt 1/5: Waiting 5s... (503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently exp)
Attempt 1/5: Waiting 5s... (503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently exp)
Attempt 1/5: Waiting 5s... (503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently exp)
Attempt 1/5: Waiting 5s... (503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently exp)
Attempt 1/5: Waiting 5s... (503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently exp)
Attempt 1/5: Waiting 5s... (503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is cur

### Sample Testing

In [28]:
# Load in the tests
ddi_class_1500_sample = pd.read_csv("C:\\Users\\ashto\\ddi-prediction\\data\\sample\\drug_class_w_gemini_2.5_flash_sample_1500_paid.csv")

In [29]:
ddi_class_1500_sample.head()

,drug1_id,drug1_name,drug2_id,drug2_name,description,pair_key,classification,Reason
0,DB00263,Sulfisoxazole,DB17171,Ziftomenib,The risk or severity of QTc prolongation can b...,"('DB00263', 'DB17171')",Cardiovascular Effects,The interaction describes an increased risk of...
1,DB00390,Digoxin,DB11090,Potassium nitrate,Potassium nitrate may decrease the excretion r...,"('DB00390', 'DB11090')",Dosage Adjustment Required,The interaction describes a decreased excretio...
2,DB00751,Epinastine,DB11830,Mocetinostat,The risk or severity of QTc prolongation can b...,"('DB00751', 'DB11830')",Cardiovascular Effects,The interaction increases the risk or severity...
3,DB00313,Valproic acid,DB12883,Eltoprazine,The risk or severity of CNS depression can be ...,"('DB00313', 'DB12883')",CNS/Neurological Effects,The interaction explicitly states an increased...
4,DB01166,Cilostazol,DB11967,Binimetinib,The metabolism of Binimetinib can be decreased...,"('DB01166', 'DB11967')",Metabolic/Absorption Interference,The interaction describes a decrease in the me...


### Sample distribution analysis

In [30]:

# Combine all samples into one dataframe for analysis
all_samples = ddi_class_1500_sample

print(f"Total samples: {len(all_samples)}")
print(f"\nDataframe shape: {all_samples.shape}")
print(f"\nColumns: {all_samples.columns.tolist()}")


Total samples: 1500

Dataframe shape: (1500, 8)

Columns: ['drug1_id', 'drug1_name', 'drug2_id', 'drug2_name', 'description', 'pair_key', 'classification', 'Reason']


In [31]:

# Distribution Analysis
# Count classifications
classification_counts = all_samples['classification'].value_counts()
print("Classification Distribution:")
print(classification_counts)
print(f"\nPercentages:")
print(classification_counts / len(all_samples) * 100)


Classification Distribution:
classification
Metabolic/Absorption Interference      648
Therapeutic Efficacy Alteration        189
CNS/Neurological Effects               177
Cardiovascular Effects                 157
General/Unspecified Adverse Effects    103
Hematological Effects                   53
Bleeding/Hemorrhage Risk                41
Clinical Metabolic Dysregulation        32
Fluid & Electrolyte Imbalance           29
Organ Toxicity                          23
Positive/Synergistic                    22
Dosage Adjustment Required              13
Gastrointestinal                         8
Immunological/Hypersensitivity           5
Name: count, dtype: int64

Percentages:
classification
Metabolic/Absorption Interference      43.200000
Therapeutic Efficacy Alteration        12.600000
CNS/Neurological Effects               11.800000
Cardiovascular Effects                 10.466667
General/Unspecified Adverse Effects     6.866667
Hematological Effects                   3.533333
Bleed

In [32]:
# Create visualizations for multi-class distribution
color_map = {
    'Bleeding/Hemorrhage Risk': '#D62728',  # Red
    'Organ Toxicity': '#FF7F0E',  # Orange
    'CNS/Neurological Effects': '#9467BD',  # Purple
    'Dosage Adjustment Required': '#1F77B4',  # Blue
    'Metabolic/Absorption Interference': '#17BECF',  # Cyan
    'Other Adverse Effects': '#E377C2',  # Pink
    'Positive/Synergistic': '#2CA02C',  # Green
    'No Significant Interaction': '#7F7F7F'  # Gray
}

fig1 = px.bar(
    x=classification_counts.index,
    y=classification_counts.values,
    labels={'x': 'DDI Type', 'y': 'Count'},
    title='Distribution of Drug-Drug Interaction Types',
    color=classification_counts.index,
    color_discrete_map=color_map
)
fig1.update_layout(showlegend=False, height=500, xaxis_tickangle=-45)
fig1.show()

# Pie chart
fig2 = px.pie(
    labels=classification_counts.index,
    values=classification_counts.values,
    title='DDI Type Distribution (%)',
    color=classification_counts.index,
    color_discrete_map=color_map
)
fig2.show()



In [33]:
# DETAILED ANALYSIS: Document prompt, clustering patterns, and deviations


# Define run timestamp for organized output
run_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
run_folder = f"C:\\Users\\ashto\\ddi-prediction\\data\\raw\\lmm_classifcation_runs\\run_{run_timestamp}"

# Create run folder if it doesn't exist
os.makedirs(run_folder, exist_ok=True)

# Save plots to run folder (fig1 and fig2 created in previous cell)
if 'fig1' in locals():
    fig1_path = os.path.join(run_folder, "08_distribution_bar_chart.png")
    fig1.write_image(fig1_path, width=1000, height=600)
    print(f"✓ Bar chart saved to: {fig1_path}")

if 'fig2' in locals():
    fig2_path = os.path.join(run_folder, "09_distribution_pie_chart.png")
    fig2.write_image(fig2_path, width=900, height=700)
    print(f"✓ Pie chart saved to: {fig2_path}")

print("\n" + "=" * 100)
print("DETAILED ANALYSIS: 15-CATEGORY SYSTEM PROMPT DOCUMENTATION")
print("=" * 100)

# 1. DEFINE THE EXACT SYSTEM PROMPT FROM ASYNC SECTION
SYSTEM_PROMPT_15_CATEGORIES = """You are an expert clinical pharmacologist analyzing Drug-Drug Interactions (DDIs) from DrugBank.
Your task is to classify each DDI into one of 15 specific categories based on the PRIMARY clinical mechanism.

ADVERSE EFFECT CATEGORIES (13):
1. "Bleeding/Hemorrhage Risk" - Increased bleeding risk, anticoagulant effects, platelet dysfunction
2. "Dosage Adjustment Required" - Dose modification needed but NO specific adverse effect mentioned; "requires monitoring" alone
3. "Organ Toxicity" - Hepatotoxicity, nephrotoxicity, cardiac toxicity, renal impairment, liver dysfunction (DIRECT organ damage)
4. "CNS/Neurological Effects" - Sedation, CNS depression, neurotoxicity, seizure risk, tremor, dizziness
5. "Metabolic/Absorption Interference" - PHARMACOKINETIC only: CYP450, bioavailability, absorption issues (NOT clinical metabolic outcomes)
6. "Cardiovascular Effects" - Hypertension, Hypotension, Tachycardia, Bradycardia, Orthostatic Hypotension, Arrhythmias (functional changes)
7. "Hematological Effects" - Methemoglobinemia, Neutropenia, other blood disorders (NOT bleeding/hemorrhage)
8. "Fluid & Electrolyte Imbalance" - Hyperkalemia, Hypokalemia, Hyponatremia, Water intoxication, Fluid retention
9. "Clinical Metabolic Dysregulation" - Hyperglycemia, Hypoglycemia, direct metabolic outcome (NOT pharmacokinetic interference)
10. "Immunological/Hypersensitivity" - Allergic reactions, hypersensitivity, infection risk, immune-related effects
11. "Therapeutic Efficacy Alteration" - Decreased/increased drug effectiveness, loss of efficacy
12. "Gastrointestinal" - GI irritation, nausea, vomiting, diarrhea (specific to this interaction)
13. "General/Unspecified Adverse Effects" - Vague descriptions like "adverse effects" without specific clinical outcome

BENEFICIAL/NEUTRAL CATEGORIES (2):
14. "Positive/Synergistic" - Beneficial interactions or enhanced therapeutic effects
15. "No Significant Interaction" - No clinically significant interaction

CRITICAL BOUNDARIES:
- Bleeding (category 1) ≠ Hematological Effects (category 7): Bleeding is hemorrhage risk; Hematological is other blood disorders
- Organ Toxicity (category 3) ≠ Cardiovascular (category 6): Organ Toxicity = direct organ damage; Cardiovascular = functional BP/HR changes
- Metabolic/Absorption (category 5) ≠ Clinical Metabolic (category 9): PK interference vs. clinical metabolic outcomes
- "Requires monitoring" alone = Dosage Adjustment (category 2), UNLESS specific adverse outcome is described

CLASSIFICATION RULES:
- Classify based on PRIMARY/MOST SIGNIFICANT effect
- If multiple effects: Bleeding > Organ Toxicity > CNS > Dosage > Metabolic/Absorption > Cardiovascular > Hematological > Electrolyte > Clinical Metabolic > Immunological > Efficacy > GI > Unspecified > Positive > None
- One category only per interaction
"""

# 2. DOCUMENT THE RUN CONFIGURATION
documented_analysis = f"""
{'=' * 100}
COMPREHENSIVE LLM ANALYSIS DOCUMENTATION - 15-CATEGORY SYSTEM
{'=' * 100}
Generated: {datetime.now().isoformat()}
Run ID: {run_timestamp}

PART 1: SYSTEM CONFIGURATION
─────────────────────────────
Model: gemini-2.5-flash
Temperature: 0.0 (deterministic, structured output)
Input Sample Size: Total DDIs in dataset (from 1500-sample)
Sampling Method: Random sampling (reset_index)
Classification Categories: 15 (13 adverse effect + 2 beneficial/neutral)

PART 2: EXACT SYSTEM PROMPT USED
──────────────────────────────────
{SYSTEM_PROMPT_15_CATEGORIES}

PART 3: CATEGORY STRUCTURE
───────────────────────────
Adverse Effect Categories (13):
  1. Bleeding/Hemorrhage Risk
  2. Dosage Adjustment Required
  3. Organ Toxicity
  4. CNS/Neurological Effects
  5. Metabolic/Absorption Interference
  6. Cardiovascular Effects
  7. Hematological Effects
  8. Fluid & Electrolyte Imbalance
  9. Clinical Metabolic Dysregulation
 10. Immunological/Hypersensitivity
 11. Therapeutic Efficacy Alteration
 12. Gastrointestinal
 13. General/Unspecified Adverse Effects

Beneficial/Neutral Categories (2):
 14. Positive/Synergistic
 15. No Significant Interaction

Key Improvements Over Previous 8-Category System:
- "Other Adverse Effects" replaced with 8 specific subcategories (6-13)
- Added critical boundary definitions to prevent misclassification
- Introduced priority hierarchy for multi-effect interactions
- Separated pharmacokinetic interference from clinical metabolic outcomes

PART 4: STATISTICS FROM RUN
────────────────────────────
"""

# Add distribution statistics
if 'classification_counts' in locals():
    documented_analysis += f"\nTotal samples classified: {len(all_samples)}\n\n"
    documented_analysis += "Classification Distribution:\n"
    documented_analysis += f"{'─' * 80}\n"
    documented_analysis += f"{'Category':<50s} {'Count':>10s} {'Percentage':>15s}\n"
    documented_analysis += f"{'─' * 80}\n"
    
    for classification in classification_counts.index:
        count = classification_counts[classification]
        pct = (count / len(all_samples)) * 100
        documented_analysis += f"{classification:<50s} {count:>10d} {pct:>14.2f}%\n"
    
    documented_analysis += f"{'─' * 80}\n"
    documented_analysis += f"{'TOTAL':<50s} {len(all_samples):>10d} {'100.00%':>15s}\n"
    
    # Category grouping
    adverse_effects_13 = all_samples[all_samples['classification'].isin([
        'Bleeding/Hemorrhage Risk', 'Dosage Adjustment Required', 'Organ Toxicity',
        'CNS/Neurological Effects', 'Metabolic/Absorption Interference',
        'Cardiovascular Effects', 'Hematological Effects', 'Fluid & Electrolyte Imbalance',
        'Clinical Metabolic Dysregulation', 'Immunological/Hypersensitivity',
        'Therapeutic Efficacy Alteration', 'Gastrointestinal', 'General/Unspecified Adverse Effects'
    ])]
    positive_neutral_2 = all_samples[all_samples['classification'].isin([
        'Positive/Synergistic', 'No Significant Interaction'
    ])]
    
    documented_analysis += f"\n\nCategory Groupings:\n"
    documented_analysis += f"  Adverse Effects (13 categories): {len(adverse_effects_13)} ({len(adverse_effects_13)/len(all_samples)*100:.2f}%)\n"
    documented_analysis += f"  Beneficial/Neutral (2 categories): {len(positive_neutral_2)} ({len(positive_neutral_2)/len(all_samples)*100:.2f}%)\n"

documented_analysis += f"""

PART 5: OUTPUT ARTIFACTS
─────────────────────────
All outputs saved to: {run_folder}

Files generated:
1. 01_prompt_documentation.txt - This comprehensive documentation
2. 02_classification_results.csv - Full classification results for all samples
3. 03_statistics_summary.json - Structured statistics and metadata
4. 04_category_distribution.json - Detailed distribution by category with samples
5. 05_boundary_validation_analysis.txt - LLM boundary verification and accuracy checks
6. 06_final_run_summary.txt - Complete run summary with all statistics
7. 07_execution_metadata.json - Machine-readable metadata for programmatic access
8. 08_distribution_bar_chart.png - Bar chart visualization
9. 09_distribution_pie_chart.png - Pie chart visualization

PART 6: QUALITY ASSURANCE NOTES
────────────────────────────────
- Temperature set to 0.0 for deterministic, consistent classifications
- Structured output format enforced via response_schema
- Pydantic validation ensures output matches expected format
- All samples include both classification and reasoning fields
- Rate limiting and retry logic implemented for API robustness
"""

# 3. SAVE MAIN DOCUMENTATION
doc_file = os.path.join(run_folder, "01_prompt_documentation.txt")
with open(doc_file, "w") as f:
    f.write(documented_analysis)

print(documented_analysis)
print(f"✓ Documentation saved to: {doc_file}")

# 4. SAVE STATISTICS AS JSON
if 'classification_counts' in locals():
    statistics_json = {
        'run_id': run_timestamp,
        'run_timestamp': datetime.now().isoformat(),
        'model': 'gemini-2.5-flash',
        'temperature': 0.0,
        'total_samples': len(all_samples),
        'num_categories': 15,
        'adverse_effect_categories': 13,
        'beneficial_neutral_categories': 2,
        'category_distribution': classification_counts.to_dict(),
        'grouping': {
            'adverse_effects_count': len(adverse_effects_13),
            'adverse_effects_percentage': round(len(adverse_effects_13)/len(all_samples)*100, 2),
            'beneficial_neutral_count': len(positive_neutral_2),
            'beneficial_neutral_percentage': round(len(positive_neutral_2)/len(all_samples)*100, 2)
        }
    }
    
    stats_file = os.path.join(run_folder, "03_statistics_summary.json")
    with open(stats_file, "w") as f:
        json.dump(statistics_json, f, indent=2)
    
    print(f"✓ Statistics saved to: {stats_file}")
    
    # 5. SAVE DETAILED CATEGORY DISTRIBUTION
    category_dist_json = {}
    for category in classification_counts.index:
        category_count = classification_counts[category]
        category_dist_json[category] = {
            'count': int(category_count),
            'percentage': round(category_count / len(all_samples) * 100, 2),
            'samples': all_samples[all_samples['classification'] == category][['drug1_name', 'drug2_name', 'description', 'Reason']].head(5).to_dict('records')
        }
    
    dist_file = os.path.join(run_folder, "04_category_distribution.json")
    with open(dist_file, "w") as f:
        json.dump(category_dist_json, f, indent=2)
    
    print(f"✓ Category distribution saved to: {dist_file}")
    
    # 6. SAVE CLASSIFICATIONS AS CSV
    csv_file = os.path.join(run_folder, "02_classification_results.csv")
    all_samples.to_csv(csv_file, index=False)
    print(f"✓ Classification results saved to: {csv_file}")

print(f"\n{'=' * 100}")
print(f"RUN COMPLETE - All outputs organized in: {run_folder}")
print(f"{'=' * 100}")


✓ Bar chart saved to: C:\Users\ashto\ddi-prediction\data\raw\lmm_classifcation_runs\run_20260709_203631\08_distribution_bar_chart.png
✓ Pie chart saved to: C:\Users\ashto\ddi-prediction\data\raw\lmm_classifcation_runs\run_20260709_203631\09_distribution_pie_chart.png

DETAILED ANALYSIS: 15-CATEGORY SYSTEM PROMPT DOCUMENTATION

COMPREHENSIVE LLM ANALYSIS DOCUMENTATION - 15-CATEGORY SYSTEM
Generated: 2026-07-09T20:36:35.536460
Run ID: 20260709_203631

PART 1: SYSTEM CONFIGURATION
─────────────────────────────
Model: gemini-2.5-flash
Temperature: 0.0 (deterministic, structured output)
Input Sample Size: Total DDIs in dataset (from 1500-sample)
Sampling Method: Random sampling (reset_index)
Classification Categories: 15 (13 adverse effect + 2 beneficial/neutral)

PART 2: EXACT SYSTEM PROMPT USED
──────────────────────────────────
You are an expert clinical pharmacologist analyzing Drug-Drug Interactions (DDIs) from DrugBank.
Your task is to classify each DDI into one of 15 specific categor

### PATTERN LEARNING: LLM Generates Classification Rules

Instead of classifying the entire 1.1M DDI dataset with the LLM, we'll:
1. Generate 3 non-overlapping runs of 1500 samples each (4500 total unique)
2. Use LLM to classify these 4500 samples
3. Extract standardized text patterns from these classifications
4. Apply patterns to classify remaining ~1.1M DDIs without LLM

In [26]:
import numpy as np
df = pd.read_csv("C:\\Users\\ashto\\ddi-prediction\\data\\sample\\drugbank_approved_small_2369_1129743_ddi_pairs.csv") # Load in the csv

# Step 1: Generate 1500-sample dataset for pattern learning
sample_size = 1500
df_pattern_learning = df.sample(n=sample_size, random_state=42).reset_index(drop=True)
classified_indices = set(df_pattern_learning.index)

print(f"Pattern learning dataset created")
print(f"  Total samples: {len(df_pattern_learning)}")


Pattern learning dataset created
  Total samples: 1500


In [27]:

# Step 2: Classify the learning dataset (1500 samples) with LLM
async def classify_pattern_learning_dataset():
    """Classify the pattern learning dataset (1500 samples)"""
    descriptions = df_pattern_learning['description'].tolist()
    all_results = []
    
    print(f"Classifying {len(descriptions)} samples for pattern learning...")
    
    # Process in batches
    batch_size = 50
    num_batches = (len(descriptions) + batch_size - 1) // batch_size
    
    for batch_num in range(num_batches):
        start_idx = batch_num * batch_size
        end_idx = min((batch_num + 1) * batch_size, len(descriptions))
        batch_descs = descriptions[start_idx:end_idx]
        
        print(f"  Batch {batch_num + 1}/{num_batches}: Processing {len(batch_descs)} items...")
        
        tasks = [classify_interaction_async(desc) for desc in batch_descs]
        batch_results = await asyncio.gather(*tasks, return_exceptions=True)
        all_results.extend(batch_results)
        
        if batch_num < num_batches - 1:
            await asyncio.sleep(0.5)
    
    # Assign results
    for idx, result in enumerate(all_results):
        if isinstance(result, Exception):
            df_pattern_learning.loc[idx, 'llm_classification'] = 'ERROR'
            df_pattern_learning.loc[idx, 'llm_reasoning'] = str(result)[:200]
        else:
            df_pattern_learning.loc[idx, 'llm_classification'] = result.classification
            df_pattern_learning.loc[idx, 'llm_reasoning'] = result.reasoning
    
    return df_pattern_learning

# Run classification
try:
    loop = asyncio.get_running_loop()
except RuntimeError:
    df_pattern_learning = asyncio.run(classify_pattern_learning_dataset())
else:
    df_pattern_learning = await classify_pattern_learning_dataset()

print("\n✓ Pattern learning dataset classified")
print(f"\nClassification Distribution:")
print(df_pattern_learning['llm_classification'].value_counts())


Classifying 1500 samples for pattern learning...
  Batch 1/30: Processing 50 items...
  Batch 2/30: Processing 50 items...
  Batch 3/30: Processing 50 items...
  Batch 4/30: Processing 50 items...
  Batch 5/30: Processing 50 items...
  Batch 6/30: Processing 50 items...
  Batch 7/30: Processing 50 items...
  Batch 8/30: Processing 50 items...
  Batch 9/30: Processing 50 items...
  Batch 10/30: Processing 50 items...
  Batch 11/30: Processing 50 items...
  Batch 12/30: Processing 50 items...
  Batch 13/30: Processing 50 items...
  Batch 14/30: Processing 50 items...
  Batch 15/30: Processing 50 items...
  Batch 16/30: Processing 50 items...
  Batch 17/30: Processing 50 items...
  Batch 18/30: Processing 50 items...
  Batch 19/30: Processing 50 items...
  Batch 20/30: Processing 50 items...
  Batch 21/30: Processing 50 items...
  Batch 22/30: Processing 50 items...
  Batch 23/30: Processing 50 items...
  Batch 24/30: Processing 50 items...
  Batch 25/30: Processing 50 items...
  Batch 26

In [28]:
df_pattern_learning.to_csv("C:\\Users\\ashto\\ddi-prediction\\data\\raw\\df_pattern_learning.csv", index=False)

In [29]:
df_pattern_learning = pd.read_csv("C:\\Users\\ashto\\ddi-prediction\\data\\raw\\df_pattern_learning.csv")

# Step 3: Ask LLM to develop REGEX patterns from the 1500 classified samples
async def develop_regex_patterns():
    """Ask LLM to analyze 1500 classified samples and create regex/pattern rules for binary classification"""
    
    # Get classified samples with their descriptions and reasoning
    classified_samples = df_pattern_learning[df_pattern_learning['llm_classification'] != 'ERROR'].copy()
    
    # Build category examples with full context
    categories = classified_samples['llm_classification'].unique()
    category_data = {}
    
    for category in categories:
        cat_samples = classified_samples[classified_samples['llm_classification'] == category]
        # Get up to 20 examples per category
        examples = []
        for idx, row in cat_samples.head(20).iterrows():
            examples.append({
                'description': row['description'][:200],
                'reasoning': row['llm_reasoning'][:100]
            })
        category_data[category] = examples
    
    # Format examples for LLM
    examples_text = ""
    for category, examples in category_data.items():
        examples_text += f"\n\n### {category}\n"
        for i, ex in enumerate(examples, 1):
            examples_text += f"\n{i}. Description: {ex['description']}\n   Reasoning: {ex['reasoning']}\n"
    
    user_prompt = f"""You are analyzing {len(classified_samples)} drug-drug interaction descriptions that I've already classified as either "Negative" or "Positive".

Here are sample descriptions grouped by their classification with the reasoning:

{examples_text}

Now, create a REGEX PATTERN MATCHING STRATEGY for binary classification.

For each category (Negative and Positive), identify:
1. MUST_HAVE_PATTERNS - Regex patterns that strongly indicate this category (high confidence)
2. COMMON_TERMS - Key terms/phrases that often appear in this category
3. NEGATIVE_INDICATORS - Patterns that suggest this is the OTHER category (should exclude/lower confidence)
4. PRIORITY - Relative priority (1-2, where 1 = highest)

Return ONLY valid JSON with this structure (no markdown, no code blocks):
{{
  "Negative": {{
    "must_have": ["regex_pattern1", "regex_pattern2", ...],
    "common_terms": ["term1", "term2", ...],
    "negative_indicators": ["pattern_to_exclude1", ...],
    "priority": 1
  }},
  "Positive": {{
    "must_have": ["regex_pattern1", ...],
    "common_terms": ["term1", ...],
    "negative_indicators": ["pattern_to_exclude1", ...],
    "priority": 2
  }}
}}

Focus on:
- Clinical terminology that distinguishes harmful vs beneficial effects
- Word patterns that indicate adverse outcomes vs positive outcomes
- Key indicators like "adverse", "toxicity", "risk", "increased" (Negative) vs "efficacy", "benefit", "no significant" (Positive)
- Make patterns that work for real text (40-300 chars)"""
    
    print("Asking LLM to develop regex pattern strategy from 1500 classified samples...")
    print(f"Analyzing {len(classified_samples)} samples across {len(categories)} categories")
    
    try:
        loop = asyncio.get_event_loop()
        result = await loop.run_in_executor(None, lambda: client.models.generate_content(
            model='gemini-2.5-flash',
            contents=user_prompt,
            config=types.GenerateContentConfig(
                temperature=0.1,  # Low temp for consistent patterns
                response_mime_type="application/json",
            ),
        ))
        
        response_text = result.text
        print(f"✓ Pattern strategy generated ({len(response_text)} chars)")
        
        # Parse JSON
        import json
        try:
            patterns_dict = json.loads(response_text)
            print(f"✓ Successfully parsed {len(patterns_dict)} categories")
            return patterns_dict
        except json.JSONDecodeError as e:
            print(f"ERROR: Failed to parse JSON: {e}")
            print(f"Response preview: {response_text[:500]}")
            return None
    except Exception as e:
        print(f"Error developing pattern strategy: {e}")
        import traceback
        traceback.print_exc()
        return None

# Develop patterns
try:
    loop = asyncio.get_running_loop()
except RuntimeError:
    regex_patterns = asyncio.run(develop_regex_patterns())
else:
    regex_patterns = await develop_regex_patterns()

if regex_patterns:
    print("\n✓ Regex pattern strategy developed from LLM analysis")
    print("\nPattern strategy preview:")
    for cat, patterns in regex_patterns.items():
        print(f"\n{cat}:")
        print(f"  Priority: {patterns.get('priority', '?')}")
        print(f"  Must-have patterns: {patterns.get('must_have', [])[:3]}")
        print(f"  Common terms: {patterns.get('common_terms', [])[:5]}")
        print(f"  Negative indicators: {patterns.get('negative_indicators', [])[:3]}")
else:
    print("WARNING: Failed to develop pattern strategy")
    regex_patterns = {}

Asking LLM to develop regex pattern strategy from 1500 classified samples...
Analyzing 1500 samples across 2 categories
✓ Pattern strategy generated (1503 chars)
✓ Successfully parsed 2 categories

✓ Regex pattern strategy developed from LLM analysis

Pattern strategy preview:

Negative:
  Priority: 1
  Must-have patterns: ['risk (or severity )?of (adverse effects|toxicity|methemoglobinemia|hypotension)', '(decrease|reduced|lower) (excretion rate|metabolism|therapeutic efficacy|antihypertensive activities|serum level|efficacy|effectiveness)', '(higher|increased) serum level']
  Common terms: ['decrease', 'increased risk', 'severity', 'higher serum level', 'reduced effectiveness']
  Negative indicators: ['(increase|increased) (the )?therapeutic efficacy', 'beneficial outcome']

Positive:
  Priority: 2
  Must-have patterns: ['(increase|increased) (the )?therapeutic efficacy', 'beneficial outcome']
  Common terms: ['increase', 'increased', 'therapeutic efficacy', 'beneficial', 'outcome']


In [32]:
# Manual Positive/Synergistic regex patterns for validation
import re

print("="*80)
print("MANUAL POSITIVE INTERACTION PATTERNS")
print("="*80)

# Positive/Synergistic patterns
SYNERGY_PATTERN = re.compile(
    r'\b(therapeutic|clinical|antimicrobial|antibacterial)\b.*?\b(synergy|synergistic|enhanced|augmented|potentiated|optimized)\b', 
    re.IGNORECASE
)
MITIGATION_PATTERN = re.compile(
    r'\b(reduce|decrease|mitigate|prevent|offset|counteract)\b.*?\b(adverse effect|side effect|toxicity|risk of)\b', 
    re.IGNORECASE
)
MANAGEMENT_PATTERN = re.compile(
    r'\b(beneficial|indicated|recommended|co-administered|adjunct therapy)\b.*?\b(management|treatment|regimen)\b', 
    re.IGNORECASE
)

print("\nPattern definitions:")
print(f"  1. SYNERGY_PATTERN - matches therapeutic/clinical synergistic effects")
print(f"  2. MITIGATION_PATTERN - matches risk/adverse effect reduction")
print(f"  3. MANAGEMENT_PATTERN - matches beneficial management/treatment patterns")

# Test patterns on positive samples
if 'positive_samples' in locals() and len(positive_samples) > 0:
    print(f"\nTesting patterns on {len(positive_samples)} positive samples:\n")
    
    pattern_hits = {
        'synergy': 0,
        'mitigation': 0,
        'management': 0,
        'any_pattern': 0
    }
    
    test_results = []
    
    for idx, (_, row) in enumerate(positive_samples.iterrows()):
        desc = row.get('description', '')
        synergy_match = SYNERGY_PATTERN.search(desc) is not None
        mitigation_match = MITIGATION_PATTERN.search(desc) is not None
        management_match = MANAGEMENT_PATTERN.search(desc) is not None
        
        has_any = synergy_match or mitigation_match or management_match
        
        if has_any:
            pattern_hits['any_pattern'] += 1
        if synergy_match:
            pattern_hits['synergy'] += 1
        if mitigation_match:
            pattern_hits['mitigation'] += 1
        if management_match:
            pattern_hits['management'] += 1
        
        test_results.append({
            'idx': idx,
            'description': desc[:80],
            'synergy': synergy_match,
            'mitigation': mitigation_match,
            'management': management_match
        })
    
    # Show statistics
    print(f"Pattern Match Statistics (out of {len(positive_samples)} positive samples):")
    print(f"  Synergy Pattern matches: {pattern_hits['synergy']} ({pattern_hits['synergy']/len(positive_samples)*100:.1f}%)")
    print(f"  Mitigation Pattern matches: {pattern_hits['mitigation']} ({pattern_hits['mitigation']/len(positive_samples)*100:.1f}%)")
    print(f"  Management Pattern matches: {pattern_hits['management']} ({pattern_hits['management']/len(positive_samples)*100:.1f}%)")
    print(f"  Any Pattern matches: {pattern_hits['any_pattern']} ({pattern_hits['any_pattern']/len(positive_samples)*100:.1f}%)")
    
    # Show examples of pattern matches
    print(f"\n{'='*80}")
    print("SAMPLE MATCHES FOR EACH PATTERN\n")
    
    for pattern_name in ['synergy', 'mitigation', 'management']:
        matching_results = [r for r in test_results if r[pattern_name]]
        if matching_results:
            print(f"{pattern_name.upper()} PATTERN - {len(matching_results)} matches:")
            for res in matching_results[:2]:
                print(f"  [{res['idx']}] {res['description']}...")
            print()
    
    print(f"{'='*80}")
    print("RECOMMENDATION:")
    if pattern_hits['any_pattern'] / len(positive_samples) > 0.5:
        print(f"✓ These patterns are EFFECTIVE for positive interactions ({pattern_hits['any_pattern']/len(positive_samples)*100:.1f}% coverage)")
        print("  Consider incorporating into the binary classifier")
    else:
        print(f"⚠ Pattern coverage is LOW ({pattern_hits['any_pattern']/len(positive_samples)*100:.1f}%)")
        print("  May need additional patterns or refinement")
else:
    print("\nWARNING: Positive samples not loaded. Run previous cell first.")


MANUAL POSITIVE INTERACTION PATTERNS

Pattern definitions:
  1. SYNERGY_PATTERN - matches therapeutic/clinical synergistic effects
  2. MITIGATION_PATTERN - matches risk/adverse effect reduction
  3. MANAGEMENT_PATTERN - matches beneficial management/treatment patterns



In [33]:
# Step 4: Build a pattern-based binary classifier from the extracted regex patterns
import re

print("Building pattern-based binary classifier from extracted regex patterns...\n")

if regex_patterns:
    def create_pattern_classifier(patterns_dict):
        """Create a binary classifier function using regex patterns"""
        
        # Compile all regex patterns for efficiency, skipping invalid patterns
        compiled_patterns = {}
        skipped_patterns = []
        
        for category, patterns_info in patterns_dict.items():
            must_have_compiled = []
            negative_compiled = []
            
            # Compile must-have patterns, skip invalid ones
            for p in patterns_info.get('must_have', []):
                try:
                    must_have_compiled.append(re.compile(p, re.IGNORECASE))
                except Exception as e:
                    skipped_patterns.append((category, 'must_have', p, str(e)))
                    print(f"⚠ Skipped invalid regex in {category}: {p[:50]}... ({str(e)[:40]})")
            
            # Compile negative indicators, skip invalid ones
            for p in patterns_info.get('negative_indicators', []):
                try:
                    negative_compiled.append(re.compile(p, re.IGNORECASE))
                except Exception as e:
                    skipped_patterns.append((category, 'negative_indicators', p, str(e)))
                    print(f"⚠ Skipped invalid regex in {category}: {p[:50]}... ({str(e)[:40]})")
            
            compiled_patterns[category] = {
                'must_have': must_have_compiled,
                'common_terms': patterns_info.get('common_terms', []),
                'negative_indicators': negative_compiled,
                'priority': patterns_info.get('priority', 99)
            }
        
        def classify_ddi(description):
            """
            Classify a DDI description as Negative or Positive using regex pattern matching.
            Returns: (category_name, confidence_score)
            """
            description_lower = description.lower()
            scores = {}
            
            # Calculate scores for each category
            for category, patterns in compiled_patterns.items():
                score = 0.0
                matches = 0
                
                # Check must-have patterns (high weight)
                for regex_pattern in patterns['must_have']:
                    try:
                        if regex_pattern.search(description_lower):
                            score += 3.0  # High weight
                            matches += 1
                    except Exception:
                        pass  # Skip if pattern fails
                
                # Check common terms (medium weight)
                for term in patterns['common_terms']:
                    if term.lower() in description_lower:
                        score += 1.0  # Medium weight
                        matches += 1
                
                # Check negative indicators (subtract)
                for regex_pattern in patterns['negative_indicators']:
                    try:
                        if regex_pattern.search(description_lower):
                            score -= 2.0  # Negative weight
                    except Exception:
                        pass  # Skip if pattern fails
                
                scores[category] = {
                    'score': max(0, score),
                    'matches': matches,
                    'priority': patterns['priority']
                }
            
            # Find best match
            if not scores or all(s['score'] <= 0 for s in scores.values()):
                # Default to Positive if no strong signals (safer default)
                return ("Positive", 0.0)
            
            # Sort by: score (descending), then by priority (ascending)
            best_category = max(
                scores.items(),
                key=lambda x: (x[1]['score'], -x[1]['priority'])
            )[0]
            
            best_score = scores[best_category]['score']
            # Normalize confidence to 0-1
            confidence = min(1.0, best_score / 5.0)
            
            return (best_category, confidence)
        
        return classify_ddi
    
    # Create the classifier
    classify_ddi = create_pattern_classifier(regex_patterns)
    print("✓ Pattern-based binary classifier created successfully")
    
    # Count patterns per category
    print(f"\nPattern statistics:")
    print(f"{'Category':<20} {'Must-Have':>12} {'Common':>10} {'Negative':>10}")
    print(f"{'-'*52}")
    for cat, patterns in regex_patterns.items():
        print(f"{cat:<20} {len(patterns.get('must_have', []))>11} {len(patterns.get('common_terms', []))>10} {len(patterns.get('negative_indicators', []))>10}")
    
    # Test on a few samples
    print("\n\nTesting pattern classifier on first 5 samples:")
    for i in range(min(5, len(df_pattern_learning))):
        desc = df_pattern_learning.iloc[i]['description']
        llm_actual = df_pattern_learning.iloc[i]['llm_classification']
        
        try:
            pred_class, conf = classify_ddi(desc)
            match = "✓" if pred_class == llm_actual else "✗"
            print(f"{match} Sample {i+1}: Actual={llm_actual}, Predicted={pred_class} (conf: {conf:.2f})")
        except Exception as e:
            print(f"✗ Sample {i+1}: Error - {str(e)[:50]}")
else:
    print("ERROR: No regex patterns available to create classifier")
    classify_ddi = None

Building pattern-based binary classifier from extracted regex patterns...

✓ Pattern-based binary classifier created successfully

Pattern statistics:
Category                Must-Have     Common   Negative
----------------------------------------------------
Negative             False True False
Positive             False False False


Testing pattern classifier on first 5 samples:
✓ Sample 1: Actual=Negative, Predicted=Negative (conf: 1.00)
✓ Sample 2: Actual=Negative, Predicted=Negative (conf: 0.80)
✗ Sample 3: Actual=Negative, Predicted=Positive (conf: 0.40)
✓ Sample 4: Actual=Negative, Predicted=Negative (conf: 1.00)
✓ Sample 5: Actual=Negative, Predicted=Negative (conf: 0.20)


In [34]:
# Step 5: Evaluate the LLM-developed classifier on the 1500 learning set
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')

print("Evaluating LLM-developed binary classifier...\n")

if classify_ddi:
    # Apply classifier to all samples
    print("Applying classifier to all 1500 learning samples...")
    predictions = []
    confidences = []
    errors = 0
    
    for idx, row in df_pattern_learning.iterrows():
        try:
            desc = row['description']
            pred_class, conf = classify_ddi(desc)
            predictions.append(pred_class)
            confidences.append(conf)
        except Exception as e:
            predictions.append('ERROR')
            confidences.append(0.0)
            errors += 1
    
    if errors > 0:
        print(f"⚠ {errors} errors during classification")
    
    df_pattern_learning['strategy_classification'] = predictions
    df_pattern_learning['strategy_confidence'] = confidences
    
    # Calculate accuracy (only on non-ERROR rows)
    valid_samples = df_pattern_learning[
        (df_pattern_learning['llm_classification'] != 'ERROR') & 
        (df_pattern_learning['strategy_classification'] != 'ERROR')
    ]
    
    if len(valid_samples) > 0:
        accuracy = accuracy_score(
            valid_samples['llm_classification'],
            valid_samples['strategy_classification']
        )
        
        print(f"\n{'='*70}")
        print(f"BINARY CLASSIFIER ACCURACY (1500 SAMPLES)")
        print(f"{'='*70}")
        print(f"Total samples: {len(valid_samples)}")
        print(f"Accuracy: {accuracy*100:.2f}%")
        correct = (valid_samples['llm_classification'] == valid_samples['strategy_classification']).sum()
        print(f"Correct: {correct} ({correct/len(valid_samples)*100:.1f}%)")
        print(f"Incorrect: {len(valid_samples)-correct} ({(len(valid_samples)-correct)/len(valid_samples)*100:.1f}%)")
        
        # Per-category accuracy
        print(f"\n{'Category':<30} {'Accuracy':>10} {'Count':>10}")
        print(f"{'-'*50}")
        categories = sorted(valid_samples['llm_classification'].unique())
        for category in categories:
            cat_samples = valid_samples[valid_samples['llm_classification'] == category]
            cat_accuracy = accuracy_score(
                cat_samples['llm_classification'], 
                cat_samples['strategy_classification']
            )
            print(f"{category:<30} {cat_accuracy*100:>9.1f}% {len(cat_samples):>10}")
        
        # Show misclassifications
        misclassified = valid_samples[
            valid_samples['llm_classification'] != valid_samples['strategy_classification']
        ]
        if len(misclassified) > 0:
            print(f"\n{'='*70}")
            print(f"MISCLASSIFICATION EXAMPLES (first 5)")
            print(f"{'='*70}")
            for idx, (_, row) in enumerate(misclassified.head(5).iterrows()):
                print(f"\n[{idx+1}] Description: {row['description'][:75]}...")
                print(f"     LLM: {row['llm_classification']}")
                print(f"     Strategy: {row['strategy_classification']} (conf: {row['strategy_confidence']:.2f})")
    else:
        print("ERROR: No valid samples for evaluation")
else:
    print("ERROR: No classifier available for evaluation")

Evaluating LLM-developed binary classifier...

Applying classifier to all 1500 learning samples...

BINARY CLASSIFIER ACCURACY (1500 SAMPLES)
Total samples: 1500
Accuracy: 83.27%
Correct: 1249 (83.3%)
Incorrect: 251 (16.7%)

Category                         Accuracy      Count
--------------------------------------------------
Negative                            83.0%       1476
Positive                           100.0%         24

MISCLASSIFICATION EXAMPLES (first 5)

[1] Description: The metabolism of Efonidipine can be increased when combined with Ivosideni...
     LLM: Negative
     Strategy: Positive (conf: 0.40)

[2] Description: The metabolism of Tolvaptan can be increased when combined with Cefradine....
     LLM: Negative
     Strategy: Positive (conf: 0.40)

[3] Description: The serum concentration of Metergoline can be increased when it is combined...
     LLM: Negative
     Strategy: Positive (conf: 0.40)

[4] Description: Bimatoprost may increase the hypotensive activities

In [ ]:
# Step 6: Apply LLM-developed binary classifier to entire dataset and save results
print("Applying LLM-developed binary strategy to entire dataset...\n")

if classify_ddi:
    # Classify all DDI pairs
    print(f"Processing {len(df):,} DDI pairs...")
    all_classifications = []
    all_confidences = []
    errors = 0
    
    for idx, row in df.iterrows():
        try:
            desc = row['description']
            pred_class, conf = classify_ddi(desc)
            all_classifications.append(pred_class)
            all_confidences.append(conf)
        except Exception as e:
            all_classifications.append('ERROR')
            all_confidences.append(0.0)
            errors += 1
        
        if (idx + 1) % 10000 == 0:
            print(f"  Processed {idx + 1:,}/{len(df):,}")
    
    if errors > 0:
        print(f"⚠ {errors} errors during classification")
    
    df['classification'] = all_classifications
    df['confidence'] = all_confidences
    
    # Show distribution
    print(f"\n{'='*70}")
    print(f"FINAL BINARY CLASSIFICATION DISTRIBUTION (All {len(df):,} DDI pairs)")
    print(f"{'='*70}")
    dist = df[df['classification'] != 'ERROR']['classification'].value_counts()
    for category, count in dist.items():
        pct = (count / len(df)) * 100
        print(f"{category:<50} {count:>10,} ({pct:>6.2f}%)")
    
    error_count = (df['classification'] == 'ERROR').sum()
    if error_count > 0:
        print(f"{'Errors':<50} {error_count:>10,} ({error_count/len(df)*100:>6.2f}%)")
    
    # Add metadata
    df['classification_method'] = 'llm-developed-binary-strategy'
    df['training_accuracy'] = accuracy if 'accuracy' in locals() else None
    
    # Save results
    output_file = "C:\\Users\\ashto\\ddi-prediction\\data\\sample\\ddi_classification_binary_complete.csv"
    df.to_csv(output_file, index=False)
    print(f"\n✓ Complete binary classification saved to: {output_file}")
    
    # Save metadata
    metadata = {
        'total_ddi_pairs': len(df),
        'pattern_learning_samples': len(df_pattern_learning),
        'training_accuracy': float(accuracy) if 'accuracy' in locals() else None,
        'average_confidence': float(df[df['classification'] != 'ERROR']['confidence'].mean()),
        'model': 'gemini-2.5-flash',
        'strategy': 'LLM-developed binary classification (Negative/Positive)',
        'classification_categories': len(dist) if 'dist' in locals() else 0,
        'errors': error_count,
    }
    
    metadata_file = "C:\\Users\\ashto\\ddi-prediction\\data\\sample\\classification_metadata_binary.json"
    with open(metadata_file, 'w') as f:
        json.dump(metadata, f, indent=2)
    print(f"✓ Metadata saved to: {metadata_file}")
    
    print(f"\n{'='*70}")
    print(f"BINARY CLASSIFICATION COMPLETE")
    print(f"{'='*70}")
    print(f"Total DDI pairs classified: {len(df):,}")
    if 'accuracy' in locals():
        print(f"Training accuracy: {accuracy*100:.2f}%")
    print(f"Average confidence: {df[df['classification'] != 'ERROR']['confidence'].mean():.2f}")
else:
    print("ERROR: No classifier available - cannot process dataset")

In [18]:


# Analyze description uniqueness in the 1500 learning samples
print("="*70)
print("DESCRIPTION UNIQUENESS ANALYSIS")
print("="*70)

# Load the pattern learning data if not already in memory
if 'df_pattern_learning' not in locals():
    df_pattern_learning = pd.read_csv("C:\\Users\\ashto\\ddi-prediction\\data\\raw\\df_pattern_learning.csv")

total_samples = len(df_pattern_learning)
unique_descriptions = df_pattern_learning['description'].nunique()
duplicate_descriptions = total_samples - unique_descriptions
duplicate_pct = (duplicate_descriptions / total_samples) * 100

print(f"\nTotal samples: {total_samples:,}")
print(f"Unique descriptions: {unique_descriptions:,}")
print(f"Duplicate descriptions: {duplicate_descriptions:,} ({duplicate_pct:.2f}%)")
print(f"Uniqueness ratio: {unique_descriptions/total_samples*100:.2f}%")

# Find the most repeated descriptions
print(f"\n{'='*70}")
print("TOP 10 MOST REPEATED DESCRIPTIONS")
print(f"{'='*70}")

desc_counts = df_pattern_learning['description'].value_counts()
for i, (desc, count) in enumerate(desc_counts.head(10).items(), 1):
    if count > 1:
        print(f"\n[{i}] Appears {count} times:")
        print(f"    {desc[:100]}{'...' if len(desc) > 100 else ''}")
        
        # Show the different classifications for this description
        classifications = df_pattern_learning[df_pattern_learning['description'] == desc]['llm_classification'].unique()
        print(f"    Classifications: {classifications}")

print(f"\n{'='*70}")
print("SUMMARY")
print(f"{'='*70}")
print(f"Data standardization: {'HIGH' if duplicate_pct < 5 else 'MEDIUM' if duplicate_pct < 20 else 'LOW'}")
print(f"  - If most descriptions are unique, text is NOT standardized")
print(f"  - If many duplicates, text uses standard templates")
print(f"\nImplication for pattern matching:")
if duplicate_pct < 5:
    print(f"  ✓ Low duplication ({duplicate_pct:.1f}%) = Diverse, natural text")
    print(f"  ✓ Pattern matching will need to handle varied phrasings")
else:
    print(f"  ⚠ High duplication ({duplicate_pct:.1f}%) = Standardized templates")
    print(f"  ⚠ Pattern matching may be easier with repeated text structures")



DESCRIPTION UNIQUENESS ANALYSIS

Total samples: 1,500
Unique descriptions: 1,500
Duplicate descriptions: 0 (0.00%)
Uniqueness ratio: 100.00%

TOP 10 MOST REPEATED DESCRIPTIONS

SUMMARY
Data standardization: HIGH
  - If most descriptions are unique, text is NOT standardized
  - If many duplicates, text uses standard templates

Implication for pattern matching:
  ✓ Low duplication (0.0%) = Diverse, natural text
  ✓ Pattern matching will need to handle varied phrasings


In [ ]:
# Find all positive interactions from the 1500 learning samples
positive_samples = df_pattern_learning[df_pattern_learning['llm_classification'] == 'Positive'].copy()

print(f"{'='*80}")
print(f"POSITIVE INTERACTIONS FROM 1500 LEARNING SAMPLES")
print(f"{'='*80}")
print(f"Total positive interactions: {len(positive_samples)}")
print(f"Percentage of total: {len(positive_samples)/len(df_pattern_learning)*100:.2f}%\n")

# Display columns
print(f"{'ID':<5} {'Drug 1':<30} {'Drug 2':<30} {'Effect':<20}")
print(f"{'-'*85}")
for idx, (_, row) in enumerate(positive_samples.iterrows(), 1):
    drug1 = str(row.get('drug1_name', 'N/A'))[:28]
    drug2 = str(row.get('drug2_name', 'N/A'))[:28]
    reason = str(row.get('llm_reasoning', 'N/A'))[:18]
    print(f"{idx:<5} {drug1:<30} {drug2:<30} {reason:<20}")

# Show sample descriptions
print(f"\n{'='*80}")
print("SAMPLE POSITIVE INTERACTION DESCRIPTIONS (first 5)\n")
for idx, (_, row) in enumerate(positive_samples.head(5).iterrows(), 1):
    print(f"[{idx}] {row.get('drug1_name', 'N/A')} + {row.get('drug2_name', 'N/A')}")
    print(f"    Description: {row.get('description', 'N/A')[:100]}")
    print(f"    Reasoning: {row.get('llm_reasoning', 'N/A')[:100]}\n")